In [81]:
from local_pool_price import *
import plotly.express as px
import plotly.io


from mainnet_launch.data_fetching.get_state_by_block import _build_blocks_to_use_dont_clip
plotly.io.templates.default = None


def _build_safe_to_usdc_token_price(blocks: list[int], method: str):

    safe_price_calls = build_safe_price_calls()

    raw_safe_price_df = get_raw_state_by_blocks(safe_price_calls, blocks, ETH_CHAIN)
    safe_to_usdc_price_df = pd.DataFrame(index=raw_safe_price_df.index)
    if method == "(Token-ETH) / (USDC-ETH)":
        safe_to_usdc_price_df["DAI_safe_price"] = (
            raw_safe_price_df["DAI_to_ETH_safe_price"] / raw_safe_price_df["USDC_to_ETH_safe_price"]
        )
        safe_to_usdc_price_df["USDT_safe_price"] = (
            raw_safe_price_df["USDT_to_ETH_safe_price"] / raw_safe_price_df["USDC_to_ETH_safe_price"]
        )

    if method == "(Token-USD) / (USDC-USD)":
        safe_to_usdc_price_df["DAI_safe_price"] = (
            raw_safe_price_df["DAI_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
        )
        safe_to_usdc_price_df["USDT_safe_price"] = (
            raw_safe_price_df["USDT_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
        )

    safe_to_usdc_price_df["USDe_safe_price"] = (
        raw_safe_price_df["USDe_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
    )
    safe_to_usdc_price_df["GHO_safe_price"] = (
        raw_safe_price_df["GHO_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
    )
    safe_to_usdc_price_df["crvUSD_safe_price"] = (
        raw_safe_price_df["crvUSD_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
    )
    safe_to_usdc_price_df["USDs_safe_price"] = (
        raw_safe_price_df["USDs_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
    )
    safe_to_usdc_price_df["FRAX_safe_price"] = (
        raw_safe_price_df["FRAX_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
    )
    safe_to_usdc_price_df["sUSDe_safe_price"] = (
        raw_safe_price_df["sUSDe_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
    )
    safe_to_usdc_price_df["USDC_safe_price"] = 1.0

    return safe_to_usdc_price_df


def fetch_pool_local_token_prices(method="(Token-ETH) / (USDC-ETH)") -> pd.DataFrame:

    backing_calls = build_backing_calls()
    token_local_price = build_all_local_pool_prices()

    token_local_price_call_list = [t.calls for t in token_local_price]
    token_local_price_calls = [c for calls in token_local_price_call_list for c in calls]

    blocks = _build_blocks_to_use_dont_clip(start_block=20759360, end_block=22077110, chain=ETH_CHAIN, approx_num_blocks_per_day=24)
    df = get_raw_state_by_blocks([*backing_calls, *token_local_price_calls], blocks, ETH_CHAIN)

    safe_to_usdc_price_df = _build_safe_to_usdc_token_price(blocks, method=method)
    df = pd.concat([df, safe_to_usdc_price_df], axis=1)

    pool_spot_price_df, pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df = (
        _extract_pool_local_spot_price_df(df, token_local_price)
    )

    safe_backing_spread_df = pd.DataFrame(index=df.index)
    backing_cols = [c for c in df.columns if "backing" in c]
    for col in backing_cols:
        token = col.split("_")[0]
        safe_price_col = f"{token}_safe_price"
        if safe_price_col in df.columns:
            safe_backing_spread_df[f"{token}"] = 100 * ((df[col] - df[safe_price_col]) / df[col])

    return (
        df,
        pool_spot_price_df,
        pool_spot_price_discount_to_backing_df,
        pool_spot_price_spread_with_safe_price_df,
        safe_backing_spread_df,
    )


def _extract_pool_local_spot_price_df(df: pd.DataFrame, token_local_price: list[TokenLocalPoolPriceDetails]):
    pool_spot_price_df = pd.DataFrame(index=df.index)
    pool_spot_price_discount_to_backing_df = pd.DataFrame(index=df.index)
    pool_spot_price_spread_with_safe_price_df = pd.DataFrame(index=df.index)

    for t in token_local_price:
        cols = [c for c in df.columns if t.pool_name in c]

        for spot_price_column in cols:
            first_token = spot_price_column.split("_")[-3]
            second_token = spot_price_column.split("_")[-1]

            pool_spot_price_df[f"{t.pool_name}__{first_token}_spot_price"] = (
                df[spot_price_column] * df[f"{second_token}_safe_price"]
            )
            spot_price = pool_spot_price_df[f"{t.pool_name}__{first_token}_spot_price"]
            backing = df[f"{first_token}_backing"]
            safe_price = df[f"{first_token}_safe_price"]

            pool_spot_price_discount_to_backing_df[f"{t.pool_name}__{first_token}_spot_price_discount_to_backing"] = (
                100 * ((backing - spot_price) / backing)
            )

            pool_spot_price_spread_with_safe_price_df[f"{t.pool_name}__{first_token}_safe_spot_spread"] = 100 * (
                (spot_price - safe_price) / safe_price
            )
    return pool_spot_price_df, pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df


from mainnet_launch.constants import WORKING_DATA_DIR
# (
#     df,
#     pool_spot_price_df,
#     pool_spot_price_discount_to_backing_df,
#     pool_spot_price_spread_with_safe_price_df,
#     safe_backing_spread_df,
# ) = fetch_pool_local_token_prices()

# price_dir = WORKING_DATA_DIR / "stable_coins_price_spread"
# price_dir.mkdir(parents=True, exist_ok=True)

# # Write each DataFrame to a separate Parquet file in the price directory
# pool_spot_price_df.to_parquet(price_dir / "pool_spot_price_df.parquet")
# pool_spot_price_discount_to_backing_df.to_parquet(price_dir / "pool_spot_price_discount_to_backing_df.parquet")
# pool_spot_price_spread_with_safe_price_df.to_parquet(price_dir / "pool_spot_price_spread_with_safe_price_df.parquet")
# safe_backing_spread_df.to_parquet(price_dir / "safe_backing_spread_df.parquet")
# 1 minute

502, message='Bad Gateway', url='https://eth-mainnet.g.alchemy.com/v2/M9VWxJElEag_cu-pCMocGJ9jX7l9sWj_' [0]


In [87]:
# Define the price directory under the working data directory
price_dir = WORKING_DATA_DIR / "stable_coins_price_spread"

# Read each DataFrame from its respective Parquet file
pool_spot_price_df = pd.read_parquet(price_dir / "pool_spot_price_df.parquet")
pool_spot_price_discount_to_backing_df = pd.read_parquet(price_dir / "pool_spot_price_discount_to_backing_df.parquet")
pool_spot_price_spread_with_safe_price_df = pd.read_parquet(price_dir / "pool_spot_price_spread_with_safe_price_df.parquet")
safe_backing_spread_df = pd.read_parquet(price_dir / "safe_backing_spread_df.parquet")

In [74]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np


def render_n_day_subplots(price_df: pd.DataFrame, name: str):

    for n_days in [1, 3, 7]:
        moving_average_df = price_df.resample("1D").last().rolling(n_days).mean()

        # Set the grid dimensions
        num_cols = 3
        num_rows = 1 + (len(price_df.columns) // num_cols)
        # Create the subplots grid with subplot titles using the column names
        fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=list(moving_average_df.columns))

        # Loop over each column and add its trace to the correct subplot cell
        for i, col in enumerate(moving_average_df.columns):
            # Calculate grid position: row and column
            row = i // num_cols + 1
            col_pos = i % num_cols + 1

            # Create a line plot for the column using px.line
            temp_fig = px.line(moving_average_df, y=col)
            temp_fig.update_layout(yaxis_title="Percent")

            # Add the traces from the temporary figure into the subplot cell
            for trace in temp_fig.data:
                fig.add_trace(trace, row=row, col=col_pos)

        fig.update_yaxes(range=[moving_average_df.min().min() * 1.1, moving_average_df.max().max() * 1.1])
        fig.update_layout(
            height=200 * num_rows,
            width=500 * num_cols,
            title_text=f"{n_days} Day Moving Average {name}",
            showlegend=False,
        )
        fig.show()


# render_n_day_subplots(
#     pool_spot_price_spread_with_safe_price_df,
#     "Safe-Spot Percent Spread via (Token-ETH) / (USDC-ETH) Chainlink price route",
# )

In [88]:
spot_price_column_to_backing = {col: f"{col.split('_')[-3]}_backing" for col in pool_spot_price_df  }


{'crvUSD_USDC_pool__crvUSD_spot_price': 'crvUSD_backing',
 'crvUSD_USDC_pool__USDC_spot_price': 'USDC_backing',
 'crvUSD_USDT_pool__USDT_spot_price': 'USDT_backing',
 'crvUSD_USDT_pool__crvUSD_spot_price': 'crvUSD_backing',
 'crvUSD_GHO_pool__GHO_spot_price': 'GHO_backing',
 'crvUSD_GHO_pool__crvUSD_spot_price': 'crvUSD_backing',
 'crvUSD_USDe_pool__crvUSD_spot_price': 'crvUSD_backing',
 'crvUSD_USDe_pool__USDe_spot_price': 'USDe_backing',
 'crvUSD_FRAX_pool__crvUSD_spot_price': 'crvUSD_backing',
 'crvUSD_FRAX_pool__FRAX_spot_price': 'FRAX_backing',
 'USDe_DAI_pool__DAI_spot_price': 'DAI_backing',
 'USDe_DAI_pool__USDe_spot_price': 'USDe_backing',
 'USDe_USDC_pool__USDC_spot_price': 'USDC_backing',
 'USDe_USDC_pool__USDe_spot_price': 'USDe_backing',
 'crvUSD_sUSDe_pool__sUSDe_spot_price': 'sUSDe_backing',
 'crvUSD_sUSDe_pool__crvUSD_spot_price': 'crvUSD_backing'}

In [ ]:
# Separate plots for each stable showing x-axis time, y axis (peg and 1 day MA safe price)
# Separate plots for each stable showing x-axis time, y axis (peg and 1 day MA spot price)
# Separate plots for each stable showing x-axis time, y axis (peg and 3 day MA safe price)
# Separate plots for each stable showing x-axis time, y axis (peg and 3 day MA spot price)
# Separate plots for each stable showing x-axis time, y axis (peg and 7 day MA safe price)
# Separate plots for each stable showing x-axis time, y axis (peg and 7 day MA spot price)
    
def render_by_destination_safe_and_spot_price_charts(df, pool_spot_price_df, pool_spot_price_discount_to_backing_df:pd.DataFrame, pool_spot_price_spread_with_safe_price_df:pd.DataFrame):
    
    spot_price_column_to_backing = {col: f"{col.split('_')[-3]}_backing" for col in pool_spot_price_df  }




    for price_df, name in zip([pool_spot_price_discount_to_backing_df, pool_spot_price_spread_with_safe_price_df,], ['End of Day Spot discount', 'peg']):
        for days in [1,3,7]:
            
            moving_average_df = pool_spot_price_df.resample('1d').last().rolling(days).average()
            backing_df = df.resample('1d').last()

            # moving_average_df = price_df.resample("1D").last().rolling(days).mean()
            # peg
            num_cols = 3
            num_rows = 1 + (len(price_df.columns) // num_cols)
            # Create the subplots grid with subplot titles using the column names
            fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=list(moving_average_df.columns))

            # Loop over each column and add its trace to the correct subplot cell
            for i, col in enumerate(moving_average_df.columns):
                # Calculate grid position: row and column
                row = i // num_cols + 1
                col_pos = i % num_cols + 1

                # Create a line plot for the column using px.line
                temp_fig = px.line(moving_average_df, y=col)
                temp_fig.update_layout(yaxis_title="Percent")

                # Add the traces from the temporary figure into the subplot cell
                for trace in temp_fig.data:
                    fig.add_trace(trace, row=row, col=col_pos)

            fig.update_yaxes(range=[moving_average_df.min().min() * 1.1, moving_average_df.max().max() * 1.1])
            fig.update_layout(
                height=200 * num_rows,
                width=500 * num_cols,
                title_text=f"{days} Day Moving Average {name}",
                showlegend=False,
            )
            fig.show()


## Prob of Safe Price Premium of Discount to Peg, Spot Price Premium of Discount to Peg, Safe Spot Spread Direction

In [65]:
def make_percent_of_time_spread_was_greater_or_less_than(spread_df: pd.DataFrame, percent_values: list[float]):
    def build_percent_of_values_less_than(x):
        def my_function(column):
            return pd.Series(100 * sum([c < x for c in column]) / len(column))

        return my_function

    def build_percent_of_values_greater_than(x):
        def my_function(column):
            return pd.Series(100 * sum([c > x for c in column]) / len(column))

        return my_function

    percent_less_than_df = pd.DataFrame()

    for p in percent_values:
        # double negative
        percent_less_than_df[f"Premium > {p}%"] = spread_df.apply(build_percent_of_values_less_than(-p)).T

    percent_greater_than_df = pd.DataFrame()
    for p in percent_values:
        percent_greater_than_df[f"Discount > {p}%"] = spread_df.apply(build_percent_of_values_greater_than(p)).T

    return pd.concat([percent_less_than_df, percent_greater_than_df], axis=1).round()


# For safe-spot spread again lets use past 6 months of data, lets use safe price in the denominator.
# In a table, note the % or probability that discount > 3, 5, 10, 20+ bps. Similarly, for premiums. We should include all stables including USDs.
# spot - safe / safe
prob_spot_price_vs_safe_price = make_percent_of_time_spread_was_greater_or_less_than(
    pool_spot_price_spread_with_safe_price_df, percent_values=[0.03, 0.05, 0.1, 0.2, 0.5]
)
prob_spot_price_vs_safe_price

,Premium > 0.03%,Premium > 0.05%,Premium > 0.1%,Premium > 0.2%,Premium > 0.5%,Discount > 0.03%,Discount > 0.05%,Discount > 0.1%,Discount > 0.2%,Discount > 0.5%
crvUSD_USDC_pool__crvUSD_safe_spot_spread,23.0,11.0,4.0,0.0,0.0,19.0,11.0,4.0,1.0,0.0
crvUSD_USDC_pool__USDC_safe_spot_spread,32.0,19.0,6.0,2.0,0.0,11.0,5.0,3.0,0.0,0.0
crvUSD_USDT_pool__USDT_safe_spot_spread,48.0,44.0,39.0,33.0,15.0,42.0,42.0,38.0,31.0,12.0
crvUSD_USDT_pool__crvUSD_safe_spot_spread,45.0,42.0,39.0,32.0,15.0,44.0,42.0,38.0,32.0,15.0
crvUSD_GHO_pool__GHO_safe_spot_spread,45.0,41.0,36.0,24.0,12.0,45.0,42.0,30.0,14.0,3.0
crvUSD_GHO_pool__crvUSD_safe_spot_spread,47.0,45.0,35.0,14.0,3.0,41.0,39.0,33.0,23.0,11.0
crvUSD_USDe_pool__crvUSD_safe_spot_spread,28.0,20.0,6.0,2.0,0.0,26.0,17.0,8.0,1.0,0.0
crvUSD_USDe_pool__USDe_safe_spot_spread,37.0,26.0,11.0,2.0,0.0,19.0,12.0,5.0,2.0,0.0
crvUSD_FRAX_pool__crvUSD_safe_spot_spread,24.0,13.0,5.0,0.0,0.0,30.0,16.0,6.0,2.0,0.0
crvUSD_FRAX_pool__FRAX_safe_spot_spread,45.0,30.0,7.0,2.0,0.0,14.0,7.0,3.0,0.0,0.0


In [70]:
prob_spot_price_discount = make_percent_of_time_spread_was_greater_or_less_than(
    pool_spot_price_discount_to_backing_df, percent_values=[0, 0.1, 0.2, 0.5, 1]
)
prob_spot_price_discount

,Premium > 0%,Premium > 0.1%,Premium > 0.2%,Premium > 0.5%,Premium > 1%,Discount > 0%,Discount > 0.1%,Discount > 0.2%,Discount > 0.5%,Discount > 1%
crvUSD_USDC_pool__crvUSD_spot_price_discount_to_backing,6.0,0.0,0.0,0.0,0.0,94.0,49.0,25.0,2.0,0.0
crvUSD_USDC_pool__USDC_spot_price_discount_to_backing,32.0,3.0,0.0,0.0,0.0,68.0,6.0,2.0,0.0,0.0
crvUSD_USDT_pool__USDT_spot_price_discount_to_backing,30.0,6.0,2.0,0.0,0.0,70.0,14.0,1.0,0.0,0.0
crvUSD_USDT_pool__crvUSD_spot_price_discount_to_backing,39.0,30.0,23.0,11.0,0.0,61.0,53.0,43.0,22.0,3.0
crvUSD_GHO_pool__GHO_spot_price_discount_to_backing,30.0,6.0,1.0,1.0,0.0,70.0,52.0,39.0,11.0,0.0
crvUSD_GHO_pool__crvUSD_spot_price_discount_to_backing,29.0,17.0,11.0,7.0,4.0,71.0,48.0,25.0,8.0,3.0
crvUSD_USDe_pool__crvUSD_spot_price_discount_to_backing,14.0,1.0,0.0,0.0,0.0,86.0,46.0,23.0,4.0,0.0
crvUSD_USDe_pool__USDe_spot_price_discount_to_backing,38.0,15.0,5.0,0.0,0.0,62.0,32.0,6.0,0.0,0.0
crvUSD_FRAX_pool__crvUSD_spot_price_discount_to_backing,14.0,0.0,0.0,0.0,0.0,86.0,45.0,24.0,3.0,0.0
crvUSD_FRAX_pool__FRAX_spot_price_discount_to_backing,0.0,0.0,0.0,0.0,0.0,100.0,100.0,96.0,5.0,0.0


In [72]:
prob_safe_price_discount = make_percent_of_time_spread_was_greater_or_less_than(
    safe_backing_spread_df, percent_values=[0, 0.1, 0.2, 0.5, 1]
)
prob_safe_price_discount

,Premium > 0%,Premium > 0.1%,Premium > 0.2%,Premium > 0.5%,Premium > 1%,Discount > 0%,Discount > 0.1%,Discount > 0.2%,Discount > 0.5%,Discount > 1%
DAI,51.0,42.0,35.0,12.0,0.0,49.0,42.0,31.0,13.0,1.0
USDe,41.0,23.0,5.0,0.0,0.0,59.0,19.0,5.0,0.0,0.0
USDC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
USDT,45.0,38.0,32.0,15.0,0.0,55.0,41.0,32.0,12.0,1.0
GHO,19.0,11.0,8.0,3.0,0.0,81.0,37.0,19.0,5.0,2.0
crvUSD,13.0,1.0,0.0,0.0,0.0,87.0,50.0,22.0,3.0,0.0
USDs,51.0,21.0,12.0,4.0,2.0,46.0,28.0,20.0,5.0,2.0
FRAX,0.0,0.0,0.0,0.0,0.0,100.0,100.0,95.0,3.0,0.0
sUSDe,28.0,15.0,2.0,0.0,0.0,72.0,67.0,56.0,12.0,1.0


In [73]:
render_n_day_subplots(
    pool_spot_price_discount_to_backing_df,
    "Spot Percent Discount to backing (Token-ETH) / (USDC-ETH) Chainlink price route",
)

NameError: name 'render_n_day_subplots' is not defined

In [ ]:
# safe_to_usdc_price_df = pd.DataFrame(index=raw_safe_price_df.index)

# safe_to_usdc_price_df["DAI_to_USD_safe_price_via DIA->USD"] = raw_safe_price_df["DAI_to_USD_safe_price"]
# safe_to_usdc_price_df["DAI_to_USDC_safe_price_via  DAI->ETH->USDC"] = (
#     raw_safe_price_df["DAI_to_ETH_safe_price"] / raw_safe_price_df["USDC_to_ETH_safe_price"]
# )
# safe_to_usdc_price_df["DAI_to_USDC_safe_price_via  DAI->USD->USDC"] = (
#     raw_safe_price_df["DAI_to_USD_safe_price"] / raw_safe_price_df["USDC_to_USD_safe_price"]
# )
# safe_to_usdc_price_df["block"] = blocks
# px.line(safe_to_usdc_price_df)